In [1]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-7_D-2_To Baunia.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/IC-1_D-2_Kalshi.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-7_D-1_To Jashimuddin.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-5_D-1_To Jashimuddin.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-9_D-1_To Housebuilding.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/IC-1_D-1_Dhaka Cantonment.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-9_D-2_To Diabari.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-1_D-2_To Airport.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-2_D-1_To Jashimuddin.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-3_D-2_To Mirpur 12.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-1_D-1_To Kuril.xlsx
/kaggle/input/master-od-matrix-analysis-datase

## Step 1: Import libraries + set paths

In [2]:
import os 
import glob #glob: find multiple files using patterns (e.g., *.xlsx)
import numpy as np
import pandas as pd
from openpyxl import load_workbook # handle Excel files at the worksheet level

DATASET_DIR = "/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025"
MASTER_PATH = os.path.join(DATASET_DIR, "Master Excel.xlsx") # Instead of writing the directory, Python joins them using the correct path separator.
OUT_PATH = "/kaggle/working/Master Excel_Aggregated.xlsx"

VEH_SHEETS = [f"vehicle_type-{i}" for i in range(1, 26)]  # Creates a list of 25 sheet names
MATRIX_SIZE = 63 


## Step 2: List ONLY location files (exclude Master)

In [3]:
all_xlsx = glob.glob(os.path.join(DATASET_DIR, "*.xlsx")) # finds all Excel files (.xlsx) in that folder

location_files = []
for f in all_xlsx: # Processes each Excel file one by one
    base = os.path.basename(f) #Removes the directory path, Keeps only the file name
    if base.lower().startswith("~$"): #Excel creates temporary lock files, These are not real data files, Skip this file and move to the next one
        continue
    if os.path.abspath(f) == os.path.abspath(MASTER_PATH): #We do not want to process the master file again
        continue
    location_files.append(f)

print("Location files found:", len(location_files))
for f in location_files:
    print(" -", os.path.basename(f))


Location files found: 19
 - MC-7_D-2_To Baunia.xlsx
 - IC-1_D-2_Kalshi.xlsx
 - MC-7_D-1_To Jashimuddin.xlsx
 - MC-5_D-1_To Jashimuddin.xlsx
 - MC-9_D-1_To Housebuilding.xlsx
 - IC-1_D-1_Dhaka Cantonment.xlsx
 - MC-9_D-2_To Diabari.xlsx
 - MC-1_D-2_To Airport.xlsx
 - MC-2_D-1_To Jashimuddin.xlsx
 - MC-3_D-2_To Mirpur 12.xlsx
 - MC-1_D-1_To Kuril.xlsx
 - MC-10_D-1_To Metro Station.xlsx
 - IC-1_D-3_Balughat.xlsx
 - MC-10_D-2_To Jashimuddin.xlsx
 - MC-5_D-2_To ECB.xlsx
 - MC-8_D-2_To ECB.xlsx
 - MC-3_D-1_To Diabari.xlsx
 - MC-8_D-1_To Kalshi.xlsx
 - MC-2_D-2_To Baunia Bazar.xlsx


## Step 3: Helper: find the “O/D” anchor cell in a sheet

In [5]:
def find_matrix_anchor(df_raw: pd.DataFrame):
    s = df_raw.astype(str) # Convert everything to string, bcz Excel cells may contain text, numbers, blanks

    """applymap() → applies the function to every cell

    For each cell:
    
    Convert to string
    
    Remove extra spaces → strip()
    
    Convert to lowercase → lower()
    
    Check if it matches any known OD labels""" 
        
    mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
    coords = np.argwhere(mask.values)  # Converts mask to NumPy array 
 
    if coords.size == 0:
        raise ValueError("Could not find 'O/D' anchor cell in this sheet.")

    r, c = coords[0] # Extract first OD position, Takes the first occurrence of O/D
    return int(r), int(c) #Return row and column indices


## Step 4: Helper: extract the 63×63 OD matrix from one sheet

In [6]:
def extract_od_matrix(xlsx_path: str, sheet_name: str) -> np.ndarray:
    df_raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)

    r0, c0 = find_matrix_anchor(df_raw) #Finds the row & column index of the “O/D” cell

    # matrix starts one row below & one column right of 'O/D'
    r_start = r0 + 1
    c_start = c0 + 1

    # Extract the 63×63 block
    block = df_raw.iloc[r_start:r_start + MATRIX_SIZE, c_start:c_start + MATRIX_SIZE]

    # Convert to numeric; NaN -> 0
    """block.stack()
    → Flattens the matrix into a single column
    
    pd.to_numeric(..., errors="coerce")
    → Converts values to numbers
    → Invalid entries become NaN
    
    .unstack(fill_value=0)
    → Reshapes back to matrix
    → Replaces NaN with 0
    
    .values.astype(float)
    → Converts to a NumPy array of floats"""
    mat = pd.to_numeric(block.stack(), errors="coerce").unstack(fill_value=0).values.astype(float)

    if mat.shape != (MATRIX_SIZE, MATRIX_SIZE):
        raise ValueError(f"Matrix shape {mat.shape}, expected {(MATRIX_SIZE, MATRIX_SIZE)}")

    return mat #Returns a clean 63 × 63 NumPy array


## Step 5: Test extraction on one file, one sheet

In [8]:
# sanity check.
test_file = location_files[0]
test_sheet = "vehicle_type-1"

mat = extract_od_matrix(test_file, test_sheet)

print("Test file:", os.path.basename(test_file))
print("Sheet:", test_sheet)
print("Shape:", mat.shape)
print("Total trips in this matrix:", mat.sum())


Test file: MC-7_D-2_To Baunia.xlsx
Sheet: vehicle_type-1
Shape: (63, 63)
Total trips in this matrix: 18.0


/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])


## Step 6: Aggregate all 19 files for all 25 vehicle-type sheets

In [9]:
agg = {sh: np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=float) for sh in VEH_SHEETS}

for fpath in location_files:
    for sh in VEH_SHEETS:
        agg[sh] += extract_od_matrix(fpath, sh)

print("Aggregation complete.")
print("Example total (vehicle_type-1):", agg["vehicle_type-1"].sum()) 


/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d",

Aggregation complete.
Example total (vehicle_type-1): 523.0


/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])


## Step 7: Helper: write aggregated matrix into Master (sheet-wise)

In [11]:
def write_matrix_to_sheet(ws, mat: np.ndarray):
    # Find 'O/D' anchor in master sheet
    anchor = None
    for row in ws.iter_rows():
        for cell in row:
            if cell.value is None: 
                continue
            v = str(cell.value).strip().lower()
            if v in ["o/d", "o\\d", "od", "o / d"]: 
                anchor = (cell.row, cell.column)  # openpyxl is 1-based
                break
        if anchor:
            break

    if not anchor:
        raise ValueError(f"Could not find 'O/D' in master sheet '{ws.title}'")

    r0, c0 = anchor
    r_start = r0 + 1
    c_start = c0 + 1

    for i in range(MATRIX_SIZE):
        for j in range(MATRIX_SIZE):
            ws.cell(row=r_start + i, column=c_start + j).value = float(mat[i, j])  


## Step 8: Write all vehicle sheets into Master and save output

In [12]:
wb = load_workbook(MASTER_PATH)

for sh in VEH_SHEETS:
    ws = wb[sh]
    write_matrix_to_sheet(ws, agg[sh])

wb.save(OUT_PATH)
print("Saved aggregated master to:", OUT_PATH)

Saved aggregated master to: /kaggle/working/Master Excel_Aggregated.xlsx


## Step 9: Quick validation 

In [13]:
check = pd.read_excel(OUT_PATH, sheet_name="vehicle_type-1", header=None)
print("Read-back successful. Open the output file to confirm.") 

Read-back successful. Open the output file to confirm.


# ONE SINGLE MASTER FILE

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from openpyxl import load_workbook

MASTER_AGG_PATH = "/kaggle/working/Master Excel_Aggregated.xlsx"
OUT_TOTAL_PATH  = "/kaggle/working/OD_Total_All_Vehicles.xlsx"

VEH_SHEETS = [f"vehicle_type-{i}" for i in range(1, 26)]
MATRIX_SIZE = 63


In [ ]:
def find_matrix_anchor(df_raw: pd.DataFrame):
    s = df_raw.astype(str)
    mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
    coords = np.argwhere(mask.values)
    if coords.size == 0:
        raise ValueError("Could not find 'O/D' anchor cell.")
    r, c = coords[0]
    return int(r), int(c)


In [ ]:
def extract_od_matrix(xlsx_path: str, sheet_name: str) -> np.ndarray:
    df_raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)

    r0, c0 = find_matrix_anchor(df_raw)
    r_start = r0 + 1
    c_start = c0 + 1

    block = df_raw.iloc[r_start:r_start + MATRIX_SIZE, c_start:c_start + MATRIX_SIZE]
    mat = pd.to_numeric(block.stack(), errors="coerce").unstack(fill_value=0).values.astype(float)

    if mat.shape != (MATRIX_SIZE, MATRIX_SIZE):
        raise ValueError(f"Matrix shape {mat.shape}, expected {(MATRIX_SIZE, MATRIX_SIZE)}")

    return mat


In [ ]:
total_mat = np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=float)
 
for sh in VEH_SHEETS:
    total_mat += extract_od_matrix(MASTER_AGG_PATH, sh)

print("Total matrix created.")
print("Grand total trips (sum of all cells):", total_mat.sum())


In [ ]:
total_df = pd.DataFrame(total_mat)

with pd.ExcelWriter(OUT_TOTAL_PATH, engine="openpyxl") as writer:
    total_df.to_excel(writer, sheet_name="OD_Total_All_Vehicles", index=False, header=False)

print("Saved:", OUT_TOTAL_PATH) 

# Freight Wise OD Matrix Master File

In [15]:
import numpy as np
import pandas as pd

# -------------------------
# Paths (Kaggle)
# -------------------------
MASTER_AGG_PATH = "/kaggle/working/Master Excel_Aggregated.xlsx"
#OUT_PATH = "/kaggle/working/Truck_OD_Matrix_VehicleTypes_20-24.xlsx"
#OUT_PATH_Bus = "/kaggle/working/BUS_OD_Matrix_VehicleTypes_14-19.xlsx"
#OUT_PATH_Car = "/kaggle/working/Car_OD_Matrix_VehicleTypes_9-12.xlsx"
#OUT_PATH_NMT = "/kaggle/working/NMT_OD_Matrix_VehicleTypes_1-5.xlsx"
#OUT_PATH_CTA = "/kaggle/working/CNG,Tempo,Auto_OD_Matrix_VT_7,8,13.xlsx"
#OUT_PATH_Car2 = "/kaggle/working/Car_OD_Matrix_VehicleTypes_9,10,12,25.xlsx"
OUT_PATH_Rickshaw = "/kaggle/working/Rickshaw_OD_Matrix_VehicleTypes_3,4.xlsx"

# -------------------------
# Select only these sheets
# -------------------------
#SELECTED_SHEETS = [f"vehicle_type-{i}" for i in range(1, 6)]  # 20,21,22,23,24
SELECTED_SHEETS = [
    "vehicle_type-3",
    "vehicle_type-4"

]
MATRIX_SIZE = 63

# -------------------------
# Helpers
# -------------------------
def find_matrix_anchor(df_raw: pd.DataFrame):
    s = df_raw.astype(str)
    mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
    coords = np.argwhere(mask.values)
    if coords.size == 0:
        raise ValueError("Could not find 'O/D' anchor cell.")
    r, c = coords[0]
    return int(r), int(c)

def extract_od_matrix(xlsx_path: str, sheet_name: str) -> np.ndarray:
    df_raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)
    r0, c0 = find_matrix_anchor(df_raw)

    # Matrix starts one row below and one column right of 'O/D'
    r_start = r0 + 1
    c_start = c0 + 1

    block = df_raw.iloc[r_start:r_start + MATRIX_SIZE, c_start:c_start + MATRIX_SIZE]
    mat = pd.to_numeric(block.stack(), errors="coerce").unstack(fill_value=0).values.astype(float)

    if mat.shape != (MATRIX_SIZE, MATRIX_SIZE):
        raise ValueError(f"Matrix shape {mat.shape}, expected {(MATRIX_SIZE, MATRIX_SIZE)}")

    return mat

# -------------------------
# Sum selected vehicle types
# -------------------------
total_mat = np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=float)

for sh in SELECTED_SHEETS:
    total_mat += extract_od_matrix(MASTER_AGG_PATH, sh)

print("Selected sheets summed:", SELECTED_SHEETS)
print("Grand total (sum of all OD cells):", total_mat.sum())

# -------------------------
# Save to new Excel (one sheet only)
# -------------------------
total_df = pd.DataFrame(total_mat)

with pd.ExcelWriter(OUT_PATH_Rickshaw, engine="openpyxl") as writer:
    total_df.to_excel(writer, sheet_name="Rickshaw_VT_3,4", index=False, header=False)

print("Saved:", OUT_PATH_Rickshaw)

/tmp/ipykernel_55/595447631.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])


Selected sheets summed: ['vehicle_type-3', 'vehicle_type-4']
Grand total (sum of all OD cells): 284.0
Saved: /kaggle/working/Rickshaw_OD_Matrix_VehicleTypes_3,4.xlsx


/tmp/ipykernel_55/595447631.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])


# Exponential vehicle wise 

In [2]:
# ==============================
# OD Matrix Aggregation (Exp_VT_*  -->  vehicle_type-*)
# - Reads 19 (or more) location files whose sheets are Exp_VT_1 ... Exp_VT_25
# - Sums OD matrices across all locations for each vehicle type
# - Writes results into MASTER template whose sheets are vehicle_type-1 ... vehicle_type-25
# - Saves a new Excel file in /kaggle/working/
# ==============================

import os
import glob
import numpy as np
import pandas as pd
from openpyxl import load_workbook

# ------------------------------
# Paths (Kaggle)
# ------------------------------
DATASET_DIR = "/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025"
MASTER_PATH = os.path.join(DATASET_DIR, "Master Excel.xlsx")
OUT_PATH = "/kaggle/working/Master Excel_Aggregated.xlsx"

# ------------------------------
# OD matrix settings
# ------------------------------
MATRIX_SIZE = 63

# INPUT sheets (in location files)
INPUT_SHEETS = [f"Exp_VT_{i}" for i in range(1, 26)]        # Exp_VT_1 ... Exp_VT_25

# OUTPUT sheets (in master template)
OUTPUT_SHEETS = [f"vehicle_type-{i}" for i in range(1, 26)] # vehicle_type-1 ... vehicle_type-25


# ==============================
# Helper: find the "O/D" anchor cell in a raw dataframe
# ==============================
def find_matrix_anchor(df_raw: pd.DataFrame):
    s = df_raw.astype(str)

    mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
    coords = np.argwhere(mask.values)

    if coords.size == 0:
        raise ValueError("Could not find 'O/D' anchor cell in this sheet.")

    r, c = coords[0]  # pandas is 0-based
    return int(r), int(c)


# ==============================
# Extract OD matrix (63x63) from a sheet
# ==============================
def extract_od_matrix(xlsx_path: str, sheet_name: str) -> np.ndarray:
    df_raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)

    r0, c0 = find_matrix_anchor(df_raw)

    # matrix starts one row below & one column right of 'O/D'
    r_start = r0 + 1
    c_start = c0 + 1

    block = df_raw.iloc[r_start:r_start + MATRIX_SIZE, c_start:c_start + MATRIX_SIZE]

    # Convert to numeric; NaN -> 0 (robust to blanks/text)
    mat = pd.to_numeric(block.stack(), errors="coerce").unstack(fill_value=0).values.astype(float)

    if mat.shape != (MATRIX_SIZE, MATRIX_SIZE):
        raise ValueError(f"[{os.path.basename(xlsx_path)} | {sheet_name}] "
                         f"Matrix shape {mat.shape}, expected {(MATRIX_SIZE, MATRIX_SIZE)}")

    return mat


# ==============================
# Write OD matrix into an openpyxl worksheet at the "O/D" location
# ==============================
def write_matrix_to_sheet(ws, mat: np.ndarray):
    # Find 'O/D' anchor in master sheet (openpyxl is 1-based)
    anchor = None
    for row in ws.iter_rows():
        for cell in row:
            if cell.value is None:
                continue
            v = str(cell.value).strip().lower()
            if v in ["o/d", "o\\d", "od", "o / d"]:
                anchor = (cell.row, cell.column)
                break
        if anchor:
            break

    if not anchor:
        raise ValueError(f"Could not find 'O/D' in master sheet '{ws.title}'")

    r0, c0 = anchor
    r_start = r0 + 1
    c_start = c0 + 1

    # Write values
    for i in range(MATRIX_SIZE):
        for j in range(MATRIX_SIZE):
            ws.cell(row=r_start + i, column=c_start + j).value = float(mat[i, j])


# ==============================
# 1) Collect location files (all .xlsx except the master + temp files)
# ==============================
all_xlsx = glob.glob(os.path.join(DATASET_DIR, "*.xlsx"))

location_files = []
for f in all_xlsx:
    base = os.path.basename(f)

    # Skip Excel temp/lock files
    if base.lower().startswith("~$"):
        continue

    # Skip the master template file
    if os.path.abspath(f) == os.path.abspath(MASTER_PATH):
        continue

    location_files.append(f)

print("Location files found:", len(location_files))
for f in location_files:
    print(" -", os.path.basename(f))

if len(location_files) == 0:
    raise ValueError("No location files found. Check DATASET_DIR and file extensions.")


# ==============================
# 2) Aggregate: read Exp_VT_* from each location file, sum into vehicle_type-* buckets
#    THIS is the "MAP them — not replace them" part:
#    Exp_VT_i  -->  vehicle_type-i
# ==============================
agg = {out_sh: np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=float) for out_sh in OUTPUT_SHEETS}

for fpath in location_files:
    for idx in range(25):
        in_sh = INPUT_SHEETS[idx]     # Exp_VT_1..25  (location file)
        out_sh = OUTPUT_SHEETS[idx]   # vehicle_type-1..25 (master template)
        agg[out_sh] += extract_od_matrix(fpath, in_sh)

print("Aggregation complete.")
print("Example total (vehicle_type-1):", agg["vehicle_type-1"].sum())


# ==============================
# 3) Write results into the master workbook (vehicle_type-* sheets)
# ==============================
wb = load_workbook(MASTER_PATH)

# Optional: quick validation that master has needed sheets
missing = [sh for sh in OUTPUT_SHEETS if sh not in wb.sheetnames]
if missing:
    raise ValueError(f"Master workbook missing sheets: {missing}")

for out_sh in OUTPUT_SHEETS:
    ws = wb[out_sh]
    write_matrix_to_sheet(ws, agg[out_sh])

wb.save(OUT_PATH)
print("Saved aggregated master to:", OUT_PATH)


Location files found: 19
 - MC-7_D-2_To Baunia.xlsx
 - IC-1_D-2_Kalshi.xlsx
 - MC-7_D-1_To Jashimuddin.xlsx
 - MC-5_D-1_To Jashimuddin.xlsx
 - MC-9_D-1_To Housebuilding.xlsx
 - IC-1_D-1_Dhaka Cantonment.xlsx
 - MC-9_D-2_To Diabari.xlsx
 - MC-1_D-2_To Airport.xlsx
 - MC-2_D-1_To Jashimuddin.xlsx
 - MC-3_D-2_To Mirpur 12.xlsx
 - MC-1_D-1_To Kuril.xlsx
 - MC-10_D-1_To Metro Station.xlsx
 - IC-1_D-3_Balughat.xlsx
 - MC-10_D-2_To Jashimuddin.xlsx
 - MC-5_D-2_To ECB.xlsx
 - MC-8_D-2_To ECB.xlsx
 - MC-3_D-1_To Diabari.xlsx
 - MC-8_D-1_To Kalshi.xlsx
 - MC-2_D-2_To Baunia Bazar.xlsx


/tmp/ipykernel_55/3403904087.py:40: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/3403904087.py:40: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/3403904087.py:40: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/3403904087.py:40: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/3403904087.py:40: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["

Aggregation complete.
Example total (vehicle_type-1): 9273.0
Saved aggregated master to: /kaggle/working/Master Excel_Aggregated.xlsx


# Freight Wise OD Matrix Master File FOr Exponential vehicle type

In [7]:
import os
import numpy as np
import pandas as pd

# ------------------------------
# Paths
# ------------------------------
FINAL_MASTER_PATH = "/kaggle/working/Master Excel_Aggregated.xlsx"  # your generated master
OUT_SUM_PATH = "/kaggle/working/OD_Sum_VT2,3.xlsx"

# ------------------------------
# Settings
# ------------------------------
MATRIX_SIZE = 63
SELECTED_SHEETS = [
    "vehicle_type-2",
    #"vehicle_type-10",
    #"vehicle_type-12",
    #"vehicle_type-25",
    #"vehicle_type-18",
    "vehicle_type-3"
]

# ------------------------------
# Helper: find "O/D" anchor in raw sheet dataframe
# ------------------------------
def find_matrix_anchor(df_raw: pd.DataFrame):
    s = df_raw.astype(str)
    mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
    coords = np.argwhere(mask.values)

    if coords.size == 0:
        raise ValueError("Could not find 'O/D' anchor cell in this sheet.")

    r, c = coords[0]  # pandas is 0-based
    return int(r), int(c)

# ------------------------------
# Extract 63x63 OD matrix from a sheet
# ------------------------------
def extract_od_matrix(xlsx_path: str, sheet_name: str) -> np.ndarray:
    df_raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)

    r0, c0 = find_matrix_anchor(df_raw)

    # matrix starts one row below & one column right of 'O/D'
    r_start = r0 + 1
    c_start = c0 + 1

    block = df_raw.iloc[r_start:r_start + MATRIX_SIZE, c_start:c_start + MATRIX_SIZE]

    # Convert to numeric; NaN -> 0
    mat = pd.to_numeric(block.stack(), errors="coerce").unstack(fill_value=0).values.astype(float)

    if mat.shape != (MATRIX_SIZE, MATRIX_SIZE):
        raise ValueError(f"[{sheet_name}] Matrix shape {mat.shape}, expected {(MATRIX_SIZE, MATRIX_SIZE)}")

    return mat

# ------------------------------
# Sum selected vehicle type matrices
# ------------------------------
total = np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=float)

for sh in SELECTED_SHEETS:
    mat = extract_od_matrix(FINAL_MASTER_PATH, sh)
    total += mat
    print(f"Added {sh}, sheet total trips = {mat.sum():,.2f}")

print("\n✅ Combined matrix total trips =", f"{total.sum():,.2f}")

# ------------------------------
# Save as new Excel (one sheet only)
# ------------------------------
df_out = pd.DataFrame(total)
with pd.ExcelWriter(OUT_SUM_PATH, engine="openpyxl") as writer:
    df_out.to_excel(writer, sheet_name="Rickshaw_VT_2,3", index=False, header=False)

print("\n✅ Saved to:", OUT_SUM_PATH)


Added vehicle_type-2, sheet total trips = 99,990.00
Added vehicle_type-3, sheet total trips = 49.00

✅ Combined matrix total trips = 100,039.00

✅ Saved to: /kaggle/working/OD_Sum_VT2,3.xlsx


/tmp/ipykernel_55/1463698945.py:29: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/1463698945.py:29: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
